# ROGII GS1.30 azimuth reduced-cost screen

**Objective.** Test whether the direction-conditioned model has enough paired signal to justify the full artifact build when the production-scale builder cannot finish within the Kaggle runtime budget.

This notebook is deliberately **non-deployable**. It uses fewer wells, PF seeds, particles, bootstrap draws, and masked-prefix wells. Its outputs may only select between stopping azimuth work and rerunning the frozen full builder; they must never be attached to an inference submission.


In [ ]:
# Artifact-builder configuration. This notebook never creates a competition submission.
import json, os, platform, tempfile, time
from pathlib import Path

ARTIFACT_BUILDER_CODE_VERSION = "azimuth-builder-screen-v1-2026-08-01"
ARTIFACT_ROOT = Path("/kaggle/working/rogii-gs130-azimuth-screen-v1") if Path("/kaggle/working").exists() else Path("./rogii-gs130-azimuth-screen-v1")
MASKED_PREFIX_FRACTIONS = (0.50, 0.65, 0.75)
MASKED_PREFIX_MIN_KNOWN = 140
MASKED_PREFIX_MAX_WELLS = int(os.environ.get("MASKED_PREFIX_MAX_WELLS", "24"))  # screening only
WELL_BOOTSTRAP_DRAWS = 500
BUILDER_SEED = 20260801
IS_HIDDEN_COMPETITION_RERUN = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "0") == "1"

# The public production contract is features.json + lgb0/1/2.pkl, averaged equally.
BASELINE_MODEL_ROOTS = (
    "/kaggle/input/datasets/fleongg/rogii-claude-models-pub",
    "/kaggle/input/rogii-claude-models-pub",
    "/kaggle/input/datasets/ravaghi/wellbore-geology-prediction-artifacts",
    "/kaggle/input/wellbore-geology-prediction-artifacts",
)


## Clean GS1.30 feature machinery

The following cells are copied mechanically from the clean notebook's self-contained learned-trajectory block. Visualization and inference cells are intentionally excluded.

In [ ]:
import os, sys, glob, time, warnings, multiprocessing
from pathlib import Path
import numpy as np
import pandas as pd
from numba import njit
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from joblib import Parallel, delayed
warnings.filterwarnings("ignore")
os.environ.setdefault("SHOW_FIGS", "0")

# ---- environment / paths (Kaggle or local) -------------------------------------
def _find_data():
    for c in ["/kaggle/input/competitions/rogii-wellbore-geology-prediction",
              "/kaggle/input/rogii-wellbore-geology-prediction"]:
        if Path(c).exists() and (Path(c)/"train").exists():
            return Path(c)
    # fallback: find any mounted folder that contains a train/ directory
    for p in glob.glob("/kaggle/input/**/train", recursive=True):
        return Path(p).parent
    return Path(os.environ.get("ROGII_DATA", "."))   # local override for development

class CFG:
    DATA = _find_data()
    OUT  = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
    seed = 42
    n_splits = 5
    n_jobs = min(8, multiprocessing.cpu_count())
    # lik-PF
    PF_SEEDS = 32
    PF_PARTICLES = 300
    PF_SCALES = (3., 5., 8., 12.)
    # FAST dev (local smoke test): limit train wells & trees
    FAST = bool(int(os.environ.get("FAST", "0")))
    N_TRAIN_WELLS = int(os.environ.get("N_TRAIN_WELLS", "320"))  # deterministic UUID-sorted screen
    USE_GPU = os.environ.get("USE_GPU", "auto")
    SHOW_FIGS = os.environ.get("SHOW_FIGS", "1") == "1"   # EDA plots (on in the notebook)

FORMATIONS = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]
def _demo_well():
    """A train well with TVT + a sizable eval zone, for the EDA plots."""
    for w in sorted(p.stem.replace("__horizontal_well", "")
                    for p in (CFG.DATA/"train").glob("*__horizontal_well.csv")):
        try:
            d = pd.read_csv(CFG.DATA/"train"/f"{w}__horizontal_well.csv", usecols=["TVT", "TVT_input"])
        except Exception:
            continue
        if "TVT" in d and d.TVT.notna().any() and d.TVT_input.isna().sum() > 2000:
            return w
    return None
print("DATA:", CFG.DATA, "| OUT:", CFG.OUT, "| cores:", CFG.n_jobs, "| FAST:", CFG.FAST)

def load_well(wid, split="train"):
    base = CFG.DATA / split
    hw = pd.read_csv(base / f"{wid}__horizontal_well.csv")
    tw = pd.read_csv(base / f"{wid}__typewell.csv").sort_values("TVT")
    return hw, tw

def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a, float) - np.asarray(b, float))**2)))


In [ ]:
# ---- single particle filters (ANCC-anchored & Z-velocity-coupled), numba ---------
PF_N = 600; ANCC_N = 600
PF_MOM = 0.993; PF_VN = 0.005; PF_PN = 0.01
PF_GR_SIG_MIN = 10.; PF_GR_SIG_MAX = 60.; PF_GR_SIG_DEF = 30.
PF_GR_WIN = 5; PF_GR_WT = 0.3; PF_RESAMP = 0.5; PF_ROUGH_P = 0.2; PF_ROUGH_V = 0.003
ANCC_ALPHA = 0.998; ANCC_RN = 0.002; ANCC_PN = 0.005; ANCC_IS = 0.3; ANCC_RP = 0.1; ANCC_RR = 0.001

BEAMS = [(10,20.,144.,2,"cons"),(10,8.,64.,2,"loose"),(8,35.,220.,1,"vcons"),
         (10,14.,90.,5,"sm5"),(20,4.,36.,3,"vloose"),(12,12.,100.,3,"mid"),(15,25.,180.,2,"stiff")]

@njit(cache=True)
def _interp1(grid, v, vmin, step):
    i = int((v - vmin) / step)
    if i < 0: return grid[0]
    n = len(grid) - 1
    if i >= n: return grid[n]
    t = (v - vmin) / step - i
    return grid[i]*(1.-t) + grid[i+1]*t

@njit(cache=True)
def _resamp(pos, aux, w, N, rp, rv):
    cum = np.zeros(N+1)
    for j in range(N): cum[j+1] = cum[j]+w[j]
    u0 = np.random.uniform(0., 1./N); np2 = np.empty(N); na = np.empty(N); ci = 0
    for j in range(N):
        u = u0+j/N
        while ci < N-1 and cum[ci+1] < u: ci += 1
        np2[j] = pos[ci]+rp*np.random.randn(); na[j] = aux[ci]+rv*np.random.randn()
    return np2, na

@njit(cache=True)
def _beam_jit(sgr, tw_gr, si, BS, mc, es):
    n = len(sgr); nt = len(tw_gr); MAX = BS*6
    bidx = np.zeros(BS, np.int64); bidx[0] = si
    bcost = np.full(BS, 1e30); bcost[0] = 0.; bn = np.int64(1)
    hI = np.zeros((n, BS), np.int64); hP = np.zeros((n, BS), np.int64)
    cI = np.zeros(MAX, np.int64); cC = np.full(MAX, 1e30); cP = np.zeros(MAX, np.int64)
    for step in range(n):
        gv = sgr[step]; nc = np.int64(0)
        for bi in range(bn):
            idx = bidx[bi]; cost = bcost[bi]
            for d in range(-2, 3):
                ni = idx+d
                if ni < 0 or ni >= nt: continue
                tot = cost+(gv-tw_gr[ni])**2/es+mc*(d if d >= 0 else -d)
                fnd = np.int64(-1)
                for ci in range(nc):
                    if cI[ci] == ni: fnd = ci; break
                if fnd >= 0:
                    if tot < cC[fnd]: cC[fnd] = tot; cP[fnd] = bi
                else:
                    if nc < MAX: cI[nc] = ni; cC[nc] = tot; cP[nc] = bi; nc += 1
        kept = min(BS, nc)
        for i in range(kept):
            mi = i
            for j in range(i+1, nc):
                if cC[j] < cC[mi]: mi = j
            if mi != i:
                cI[i], cI[mi] = cI[mi], cI[i]; cC[i], cC[mi] = cC[mi], cC[i]; cP[i], cP[mi] = cP[mi], cP[i]
        hI[step, :kept] = cI[:kept]; hP[step, :kept] = cP[:kept]
        bidx[:kept] = cI[:kept]; bcost[:kept] = cC[:kept]; bn = kept
    best = np.int64(0)
    for b in range(1, bn):
        if bcost[b] < bcost[best]: best = b
    path = np.zeros(n, np.int64); b = best
    for s in range(n-1, -1, -1): path[s] = hI[s, b]; b = hP[s, b]
    return path

@njit(cache=True)
def _pf_ancc(md_v, z_v, gr_v, gg, vmin, step, gs, ls, ir, N, ALPHA, RN, PN, IS, RP, RR, RESAMP):
    pos = np.empty(N); rate = np.empty(N); w = np.ones(N)/N
    for j in range(N):
        pos[j] = ls+IS*np.random.randn(); rate[j] = ir+0.01*np.random.randn()
    pts = np.empty(len(md_v)); std_ = np.empty(len(md_v)); pm = md_v[0]-1.
    for i in range(len(md_v)):
        dm = md_v[i]-pm; dm = max(dm, 1.)
        for j in range(N):
            rate[j] = ALPHA*rate[j]+RN*np.random.randn(); pos[j] += rate[j]*dm+PN*np.random.randn()
            tvt_j = pos[j]-z_v[i]; tvt_j = max(tvt_j, vmin-50.); tvt_j = min(tvt_j, vmin+len(gg)*step+50.)
            pos[j] = tvt_j+z_v[i]
        if not np.isnan(gr_v[i]):
            ws = 0.
            for j in range(N):
                eg = _interp1(gg, pos[j]-z_v[i], vmin, step); d = (gr_v[i]-eg)/gs
                lk = max(np.exp(-0.5*d*d) if d*d < 600. else 0., 1e-300); w[j] *= lk; ws += w[j]
            if ws > 0.:
                for j in range(N): w[j] /= ws
            else:
                for j in range(N): w[j] = 1./N
        ne = 0.
        for j in range(N): ne += w[j]*w[j]
        if 1./ne < RESAMP*N:
            pos, rate = _resamp(pos, rate, w, N, RP, RR)
            for j in range(N): w[j] = 1./N
        tv = 0.
        for j in range(N): tv += w[j]*(pos[j]-z_v[i])
        pts[i] = tv; va = 0.
        for j in range(N): va += w[j]*(pos[j]-z_v[i]-tv)**2
        std_[i] = va**0.5; pm = md_v[i]
    return pts, std_

@njit(cache=True)
def _pf_z(md_v, z_v, gr_v, gr_sm_v, gg_p, gg_s, vmin, step, gs, ip, iv, beta, icpt, zsig, N,
         MOM, VN, PN, GR_WT, RP, RV, RESAMP):
    pos = np.empty(N); vel = np.empty(N); w = np.ones(N)/N
    for j in range(N):
        pos[j] = ip+0.5*np.random.randn(); vel[j] = iv+0.02*np.random.randn()
    pts = np.empty(len(md_v)); std_ = np.empty(len(md_v)); pm = md_v[0]-1.; pz = z_v[0]-1.
    for i in range(len(md_v)):
        dm = md_v[i]-pm; dm = max(dm, 1.); dzd = (z_v[i]-pz)/dm; ve = beta*dzd+icpt
        for j in range(N):
            vel[j] = MOM*vel[j]+VN*np.random.randn(); pos[j] += vel[j]*dm+PN*np.random.randn()
            pos[j] = max(pos[j], vmin-50.); pos[j] = min(pos[j], vmin+len(gg_p)*step+50.)
        if not np.isnan(gr_v[i]):
            ws = 0.
            for j in range(N):
                ep = _interp1(gg_p, pos[j], vmin, step); dp = (gr_v[i]-ep)/gs
                lp = max(np.exp(-0.5*dp*dp) if dp*dp < 600. else 0., 1e-300)
                if not np.isnan(gr_sm_v[i]):
                    es = _interp1(gg_s, pos[j], vmin, step); ds = (gr_sm_v[i]-es)/(gs*1.5)
                    lsm = max(np.exp(-0.5*ds*ds) if ds*ds < 600. else 0., 1e-300); lk = (1.-GR_WT)*lp+GR_WT*lsm
                else: lk = lp
                lk = max(lk, 1e-300); w[j] *= lk; ws += w[j]
            if ws > 0.:
                for j in range(N): w[j] /= ws
            else:
                for j in range(N): w[j] = 1./N
        ws2 = 0.
        for j in range(N):
            dv = (vel[j]-ve)/max(zsig*2., 0.005); lz = max(np.exp(-0.5*dv*dv) if dv*dv < 600. else 0., 1e-300)
            w[j] *= lz; ws2 += w[j]
        if ws2 > 0.:
            for j in range(N): w[j] /= ws2
        else:
            for j in range(N): w[j] = 1./N
        ne = 0.
        for j in range(N): ne += w[j]*w[j]
        if 1./ne < RESAMP*N:
            pos, vel = _resamp(pos, vel, w, N, RP, RV)
            for j in range(N): w[j] = 1./N
        wm = 0.
        for j in range(N): wm += w[j]*pos[j]
        pts[i] = wm; va = 0.
        for j in range(N): va += w[j]*(pos[j]-wm)**2
        std_[i] = va**0.5; pm = md_v[i]; pz = z_v[i]
    return pts, std_

def _grid(tw_tvt, tw_gr, step=0.2):
    tmin = float(tw_tvt.min()); tmax = float(tw_tvt.max())
    tvt_g = np.arange(tmin, tmax+step, step)
    return np.interp(tvt_g, tw_tvt, tw_gr).astype(np.float64), float(tmin), float(step)

def _gr_sig(hw, tw_tvt, tw_gr):
    kn = hw[hw.TVT_input.notna() & hw.GR.notna()]
    if len(kn) < 20: return float(PF_GR_SIG_DEF)
    return float(np.clip(np.std(kn.GR.values-np.interp(kn.TVT_input.values, tw_tvt, tw_gr)),
                         PF_GR_SIG_MIN, PF_GR_SIG_MAX))

def _nn(arr, v):
    i = int(np.searchsorted(arr, v, "left"))
    if i >= len(arr): return len(arr)-1
    if i > 0 and abs(arr[i-1]-v) <= abs(arr[i]-v): return i-1
    return i

def _smooth(vals, fb, r):
    s = pd.Series(vals, dtype="float32").interpolate(limit_direction="both").fillna(fb)
    return (s.rolling(r*2+1, center=True, min_periods=1).mean() if r > 0 else s).to_numpy(np.float32)

def beam_search(gr_h, tw_tvt, tw_gr, start_tvt, bs, mc, es, r):
    si = _nn(tw_tvt, start_tvt); sgr = _smooth(gr_h, float(np.nanmean(tw_gr)), r).astype(np.float64)
    return tw_tvt[_beam_jit(sgr, tw_gr.astype(np.float64), si, bs, float(mc), float(es))].astype(np.float32)

def run_pf_ancc(hw, tw_tvt, tw_gr, N=ANCC_N):
    gs = _gr_sig(hw, tw_tvt, tw_gr); kn = hw[hw.TVT_input.notna()]; ev = hw[hw.TVT_input.isna()]
    if len(ev) == 0: return np.array([]), np.array([])
    ls = float(kn.TVT_input.iloc[-1]+kn.Z.iloc[-1])
    tail = kn.tail(30); dt = np.diff(tail.TVT_input.values); dz = np.diff(tail.Z.values); dm = np.diff(tail.MD.values); m = dm > 0
    ir = float(np.median((dt+dz)[m]/dm[m])) if m.sum() >= 3 else 0.
    gg, gmin, gst = _grid(tw_tvt, tw_gr)
    pts, std = _pf_ancc(ev.MD.values.astype(np.float64), ev.Z.values.astype(np.float64), ev.GR.values.astype(np.float64),
                        gg, gmin, gst, gs, ls, ir, N, ANCC_ALPHA, ANCC_RN, ANCC_PN, ANCC_IS, ANCC_RP, ANCC_RR, PF_RESAMP)
    return pts.astype(np.float32), std.astype(np.float32)

def run_pf_z(hw, tw_tvt, tw_gr, N=PF_N):
    gs = _gr_sig(hw, tw_tvt, tw_gr); tw_s = pd.Series(tw_gr).rolling(PF_GR_WIN, center=True, min_periods=1).mean().values.astype(np.float32)
    kna = hw[hw.TVT_input.notna()]; ev = hw[hw.TVT_input.isna()]
    if len(ev) == 0: return np.array([]), np.array([])
    dz_k = np.diff(kna.Z.values); dvt = np.diff(kna.TVT_input.values); dmd_k = np.diff(kna.MD.values); m2 = dmd_k > 0
    if m2.sum() >= 10:
        vz = dz_k[m2]/dmd_k[m2]; vt = dvt[m2]/dmd_k[m2]; A = np.column_stack([vz, np.ones_like(vz)])
        c, _, _, _ = np.linalg.lstsq(A, vt, rcond=None)
        beta, icpt, zsig = float(c[0]), float(c[1]), max(float(np.std(vt-(c[0]*vz+c[1]))), 0.001)
    else: beta, icpt, zsig = -1., 0., 0.1
    t2 = kna.tail(20); dvt2 = np.diff(t2.TVT_input.values); dmd2 = np.diff(t2.MD.values); m3 = dmd2 > 0
    iv = float(np.median(dvt2[m3]/dmd2[m3])) if m3.sum() >= 3 else 0.
    gg, gmin, gst = _grid(tw_tvt, tw_gr); gs2, _, _ = _grid(tw_tvt, tw_s)
    gr_sm = hw.GR.rolling(PF_GR_WIN, center=True, min_periods=1).mean()
    pts, std = _pf_z(ev.MD.values.astype(np.float64), ev.Z.values.astype(np.float64), ev.GR.values.astype(np.float64),
                     gr_sm.loc[ev.index].values.astype(np.float64), gg, gs2, gmin, gst, gs,
                     float(kna.TVT_input.iloc[-1]), iv, beta, icpt, zsig, N,
                     PF_MOM, PF_VN, PF_PN, PF_GR_WT, PF_ROUGH_P, PF_ROUGH_V, PF_RESAMP)
    return pts.astype(np.float32), std.astype(np.float32)

def multi_scale_ncc(kgr, ktvt, hgr, hws=(8, 15, 25), stride=3):
    out = []
    for hw in hws:
        win = 2*hw+1; nk = len(kgr); nh = len(hgr)
        if nk < win+1 or nh == 0:
            out.append((np.full(nh, ktvt[-1], np.float32), np.zeros(nh, np.float32))); continue
        kg = pd.Series(kgr).rolling(5, center=True, min_periods=1).mean().values.astype(np.float32)
        hg = pd.Series(hgr).rolling(5, center=True, min_periods=1).mean().values.astype(np.float32)
        sts = np.arange(0, nk-win+1, stride, dtype=np.int32)
        if len(sts) == 0:
            out.append((np.full(nh, ktvt[-1], np.float32), np.zeros(nh, np.float32))); continue
        C = kg[sts[:, None]+np.arange(win, dtype=np.int32)[None, :]].astype(np.float32)
        Cn = (C-C.mean(1, keepdims=True))/(C.std(1, keepdims=True)+1e-6)
        hp = np.pad(hg, hw, mode="edge"); H = hp[np.arange(nh)[:, None]+np.arange(win)[None, :]].astype(np.float32)
        Hn = (H-H.mean(1, keepdims=True))/(H.std(1, keepdims=True)+1e-6)
        ncc = Hn@Cn.T/win; best = ncc.argmax(1); score = ncc.max(1).astype(np.float32)
        out.append((ktvt[np.clip(sts[best]+hw, 0, nk-1)].astype(np.float32), score))
    tvts = np.stack([o[0] for o in out], 1); scores = np.stack([o[1] for o in out], 1)
    sw = np.exp(3.*scores); sw /= sw.sum(1, keepdims=True)+1e-9
    return out, (tvts*sw).sum(1).astype(np.float32)


In [ ]:
# ---- 128-seed likelihood-weighted particle filter (the workhorse), numba ---------
@njit(cache=True, nogil=True)
def _pf_lik_allseeds(md_v, z_v, gr_v, gg, vmin, step, gs, ls, ir, N, n_seeds, seed_base,
                     MOM, VN, PN, RP, RR, RESAMP, init_spr):
    n = len(md_v); preds = np.empty((n_seeds, n)); liks = np.empty(n_seeds); tmax = vmin + len(gg)*step
    for s in range(n_seeds):
        np.random.seed(seed_base + s)
        pos = np.empty(N); rate = np.empty(N); w = np.ones(N)/N
        for j in range(N):
            pos[j] = ls + init_spr*np.random.randn(); rate[j] = ir + 0.01*np.random.randn()
        log_lik = 0.0; prev_md = md_v[0] - 1.0
        for i in range(n):
            dm = md_v[i] - prev_md
            if dm < 1.0: dm = 1.0
            for j in range(N):
                rate[j] = MOM*rate[j] + VN*np.random.randn(); pos[j] += rate[j]*dm + PN*np.random.randn()
                tvt_j = pos[j] - z_v[i]
                if tvt_j < vmin-100.: tvt_j = vmin-100.
                if tvt_j > tmax+100.: tvt_j = tmax+100.
                pos[j] = tvt_j + z_v[i]
            avg_lk = 0.0
            for j in range(N):
                eg = _interp1(gg, pos[j]-z_v[i], vmin, step); d = (gr_v[i]-eg)/gs; dd = d*d
                if dd > 600.: dd = 600.
                lk = np.exp(-0.5*dd)
                if lk < 1e-300: lk = 1e-300
                avg_lk += w[j]*lk; w[j] = w[j]*lk
            if avg_lk < 1e-300: avg_lk = 1e-300
            log_lik += np.log(avg_lk)
            ws = 0.0
            for j in range(N): ws += w[j]
            if ws > 0.0:
                for j in range(N): w[j] /= ws
            else:
                for j in range(N): w[j] = 1./N
            neff = 0.0
            for j in range(N): neff += w[j]*w[j]
            neff = 1.0/neff
            if neff < RESAMP*N:
                cum = np.empty(N); c = 0.0
                for j in range(N): c += w[j]; cum[j] = c
                u0 = np.random.uniform(0., 1./N); newpos = np.empty(N); newrate = np.empty(N); ci = 0
                for j in range(N):
                    u = u0 + j/N
                    while ci < N-1 and cum[ci] < u: ci += 1
                    newpos[j] = pos[ci] + RP*np.random.randn(); newrate[j] = rate[ci] + RR*np.random.randn()
                for j in range(N): pos[j] = newpos[j]; rate[j] = newrate[j]; w[j] = 1./N
            est = 0.0
            for j in range(N): est += w[j]*(pos[j]-z_v[i])
            preds[s, i] = est; prev_md = md_v[i]
        liks[s] = log_lik
    return preds, liks

def lik_pf(hw, tw, n_particles=CFG.PF_PARTICLES, n_seeds=CFG.PF_SEEDS, scales=CFG.PF_SCALES,
           init_spr=4.5, seed_base=0, with_quality=False):
    """Likelihood-weighted PF ensemble. Returns ({pf_scale_X: pred_eval}, ev_index[, quality])."""
    tw_s = tw.sort_values("TVT"); tw_tvt = tw_s.TVT.values.astype(float)
    tw_gr = tw_s.GR.fillna(tw_s.GR.mean()).values.astype(float)
    kn = hw[hw.TVT_input.notna()]; ev = hw[hw.TVT_input.isna()]
    if len(ev) == 0: return {}, np.array([]), {}
    last = kn.iloc[-1]; ls = float(last.TVT_input) + float(last.Z)
    tw_at_k = np.interp(kn.TVT_input.values, tw_tvt, tw_gr)
    gs = float(np.clip(np.nanstd(kn.GR.fillna(0).values - tw_at_k), 10., 60.)) * 1.3
    tail = kn.tail(30); dt = np.diff(tail.TVT_input.values); dz = np.diff(tail.Z.values); dm = np.diff(tail.MD.values); m = dm > 0
    ir = float(np.median((dt+dz)[m]/dm[m])) if m.sum() >= 3 else 0.0
    gg, gmin, gst = _grid(tw_tvt, tw_gr)
    gr_v = hw.GR.interpolate(limit_direction="both").fillna(tw_gr.mean()).values.astype(float)[ev.index]
    preds, liks = _pf_lik_allseeds(ev.MD.values.astype(float), ev.Z.values.astype(float), gr_v,
                                   gg, gmin, gst, gs, ls, ir, n_particles, n_seeds, seed_base,
                                   0.998, 0.002, 0.005, 0.1, 0.001, 0.5, init_spr)
    ln = liks - liks.max(); out = {}
    for sc in scales:
        wts = np.exp(ln/float(sc)); wts /= wts.sum(); out[f"pf_scale_{sc:g}"] = (wts[:, None]*preds).sum(0)
    out["pf_mean"] = preds.mean(0)
    q = {}
    if with_quality:
        q = {"pf_best_ll": float(liks.max())/len(ev), "pf_ll_spread": float(liks.std()),
             "pf_pt_std": preds.std(0).astype(np.float32), "pf_gr_sig": gs}
    return out, ev.index.values, q

# JIT warm-up so timings below are representative
_m = np.linspace(1, 50, 20); _z = np.zeros(20); _g = np.full(20, 50.); _gg = np.linspace(45, 55, 100)
_pf_ancc(_m, _z, _g, _gg, 45., .1, 20., 50., 0., 8, .998, .002, .005, .3, .1, .001, .5)
_pf_z(_m, _z, _g, _g, _gg, _gg, 45., .1, 20., 50., 0., -1., 0., .1, 8, .993, .005, .01, .3, .2, .003, .5)
_beam_jit(np.random.randn(30), np.random.randn(50), 25, 8, 15., 100.)
_pf_lik_allseeds(_m, _z, _g, _gg, 45., .1, 20., 50., 0., 64, 4, 0, .998, .002, .005, .1, .001, .5, 4.5)
print("trackers compiled.")

def fig_tracker_vs_truth(wid):
    import matplotlib.pyplot as plt
    hw, tw = load_well(wid); kn = hw[hw.TVT_input.notna()]; ev = hw[hw.TVT_input.isna()]
    tw_tvt = tw.TVT.to_numpy(np.float32); tw_gr = tw.GR.to_numpy(np.float32); last = float(kn.TVT_input.iloc[-1])
    pf, _ = run_pf_ancc(hw, tw_tvt, tw_gr); out, _, _ = lik_pf(hw, tw, scales=(3.,))
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(ev.MD, ev.TVT, lw=2.2, color="black", label="True TVT", zorder=5)
    ax.plot(ev.MD, np.full(len(ev), last), lw=1.1, color="gray", ls=":", label="last-known baseline")
    ax.plot(ev.MD, pf, lw=1.0, color="tab:blue", alpha=.8, label="single particle filter")
    ax.plot(ev.MD, out["pf_scale_3"], lw=1.5, color="crimson", alpha=.9, label="128-seed lik-weighted PF")
    ax.set_xlabel("MD (ft)"); ax.set_ylabel("TVT (ft)"); ax.invert_yaxis(); ax.grid(alpha=.25)
    ax.set_title(f"Well {wid}: trackers vs ground truth â€” the lik-PF resists drift"); ax.legend(loc="best")
    plt.tight_layout(); plt.show()


In [ ]:
PLANE_K = 10; DENSE_SPW = 60; DENSE_K = 20

def robust_slope(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float); m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 2 or np.std(x[m]) < 1e-6: return 0.
    return float(np.polyfit(x[m], y[m], 1)[0])

def affine_cal(kgr, tw_at_k, min_pts=20):
    v = np.isfinite(kgr) & np.isfinite(tw_at_k)
    if v.sum() < min_pts or np.std(tw_at_k[v]) < 1e-6:
        return 1., float(np.nanmean(kgr)-np.nanmean(tw_at_k)) if v.any() else 0.
    a, b = np.polyfit(tw_at_k[v], kgr[v], 1); return float(a), float(b)

def seg_b_well(ktvt, kz, form_col):
    bv = ktvt+kz-form_col; n = len(bv); b_full = float(np.median(bv))
    b_late = float(np.median(bv[max(0, n-50):])) if n >= 5 else b_full
    t1, t2 = n//3, 2*n//3
    b_early = float(np.median(bv[:max(1, t1)])) if t1 > 0 else b_full
    b_mid = float(np.median(bv[t1:max(t1+1, t2)])) if t2 > t1 else b_full
    w = np.exp(0.02*np.arange(n)); w /= w.sum()
    return b_full, b_early, b_mid, b_late, float(np.dot(w, bv))

class FormationPlaneKNN:
    def __init__(self, well_ids, data_dir):
        rows = []
        for wid in well_ids:
            try: df = pd.read_csv(data_dir/f"{wid}__horizontal_well.csv", usecols=["X","Y"]+FORMATIONS).dropna()
            except: continue
            if len(df) == 0: continue
            row = {"wid": wid, "x": float(df.X.median()), "y": float(df.Y.median())}
            for c in FORMATIONS: row[f"{c}_m"] = float(df[c].median())
            rows.append(row)
        self.df = pd.DataFrame(rows); self.wmap = {w: i for i, w in enumerate(self.df.wid)}
        xy = self.df[["x","y"]].to_numpy(); self.scale = np.where(xy.std(0) < 1e-3, 1., xy.std(0))
        self.tree = cKDTree(xy/self.scale); self.xa = self.df.x.to_numpy(); self.ya = self.df.y.to_numpy()
        self.fa = self.df[[f"{c}_m" for c in FORMATIONS]].to_numpy(np.float64)
    def impute(self, xy_q, self_wid=None, k=PLANE_K):
        q = xy_q/self.scale; nf = min(k+5, len(self.df)); dist, idx = self.tree.query(q, k=nf, workers=-1)
        if self_wid in self.wmap: dist = np.where(idx == self.wmap[self_wid], np.inf, dist)
        ordr = np.argpartition(dist, min(k-1, nf-1), 1)[:, :k]
        dk = np.take_along_axis(dist, ordr, 1); ik = np.take_along_axis(idx, ordr, 1)
        vk = np.isfinite(dk); w = np.where(vk, 1./(dk+1e-3), 0.).astype(np.float64)
        xn = self.xa[ik]; yn = self.ya[ik]; fn = self.fa[ik]; wx = w*xn; wy = w*yn
        A = np.zeros((len(q), 3, 3))
        A[:,0,0]=(wx*xn).sum(1); A[:,0,1]=(wx*yn).sum(1); A[:,0,2]=wx.sum(1)
        A[:,1,0]=A[:,0,1]; A[:,1,1]=(wy*yn).sum(1); A[:,1,2]=wy.sum(1)
        A[:,2,0]=A[:,0,2]; A[:,2,1]=A[:,1,2]; A[:,2,2]=w.sum(1)
        A[:,0,0]+=1e-9; A[:,1,1]+=1e-9; A[:,2,2]+=1e-9
        rhs = np.stack([(wx[:,:,None]*fn).sum(1), (wy[:,:,None]*fn).sum(1), (w[:,:,None]*fn).sum(1)], 1)
        try: coef = np.linalg.solve(A, rhs)
        except:
            coef = np.zeros((len(q), 3, 6))
            for r in range(len(q)):
                try: coef[r] = np.linalg.pinv(A[r])@rhs[r]
                except: pass
        Xq = xy_q[:,0]; Yq = xy_q[:,1]
        pred = (Xq[:,None]*coef[:,0,:]+Yq[:,None]*coef[:,1,:]+coef[:,2,:]).astype(np.float32)
        pred[~vk.any(1)] = self.fa.mean(0)
        return pred, np.where(vk, dk, np.inf).min(1).astype(np.float32)

class DenseANCCImputer:
    def __init__(self, well_ids, data_dir, spw=DENSE_SPW):
        xs, ys, an, wd = [], [], [], []
        for wid in well_ids:
            try: df = pd.read_csv(data_dir/f"{wid}__horizontal_well.csv", usecols=["X","Y","ANCC"]).dropna()
            except: continue
            if len(df) == 0: continue
            ix = np.linspace(0, len(df)-1, min(spw, len(df)), dtype=int); s = df.iloc[ix]
            xs.append(s.X.values); ys.append(s.Y.values); an.append(s.ANCC.values); wd.extend([wid]*len(s))
        self.xy = np.column_stack([np.concatenate(xs), np.concatenate(ys)])
        self.ancc = np.concatenate(an).astype(np.float32); self.wids = np.array(wd)
        self.scale = np.where(self.xy.std(0) < 1e-3, 1., self.xy.std(0)); self.tree = cKDTree(self.xy/self.scale)
    def impute(self, xy_q, self_wid=None, k=DENSE_K, nfetch=5000):
        xy_q = np.atleast_2d(xy_q); q = xy_q/self.scale; nf = min(nfetch, len(self.ancc))
        dist, idx = self.tree.query(q, k=nf, workers=-1)
        if self_wid: dist = np.where(self.wids[idx] == self_wid, np.inf, dist)
        ordr = np.argpartition(dist, min(k-1, nf-1), 1)[:, :k]
        dk = np.take_along_axis(dist, ordr, 1); ik = np.take_along_axis(idx, ordr, 1)
        vk = np.isfinite(dk); w = np.where(vk, 1./(dk+1e-3), 0.); sw = w.sum(1); safe = np.where(sw < 1e-9, 1., sw)
        a = self.ancc[ik]; ap = (a*w).sum(1)/safe; ap = np.where(sw < 1e-9, float(self.ancc.mean()), ap)
        var = ((a-ap[:,None])**2*w).sum(1)/safe
        return ap.astype(np.float32), np.sqrt(np.maximum(var, 0.)).astype(np.float32), np.where(vk, dk, np.inf).min(1).astype(np.float32)

_FI = None; _DI = None
ANCH_OFFS = np.array([-80,-40,-20,-10,-5,0,5,10,20,40,80], np.float32)
BEAM_OFFS = np.array([-40,-20,-10,-5,-3,0,3,5,10,20,40], np.float32)
SC_OFFS = np.array([-30,-15,-8,-4,-2,0,2,4,8,15,30], np.float32)
PF_OFFS = SC_OFFS.copy()


## Direction definition

Fit the unsigned dominant field axis using training-well X/Y surveys only; then add four well-constant, target-free features.

In [ ]:
# ---- deterministic, train-only azimuth features --------------------------------
AZIMUTH_FEATURES = ("az_dir", "az_conf", "az_cos", "az_sin")
AZIMUTH_MIN_DISPLACEMENT = 250.0
AZIMUTH_MIN_CONFIDENCE = 0.50
AZIMUTH_ENDPOINT_FRACTION = 0.05
AZIMUTH_MIN_ROWS = 20
_AZ_AXIS = None

def _trajectory_unit_vector(hw):
    """Robust start-to-end XY unit vector; target/TVT independent."""
    q = hw[["X", "Y", "MD"]].apply(pd.to_numeric, errors="coerce").dropna().sort_values("MD")
    n = len(q)
    if n < AZIMUTH_MIN_ROWS:
        return None, 0.0, "too_few_rows"
    k = max(10, int(np.ceil(AZIMUTH_ENDPOINT_FRACTION * n)))
    k = min(k, n // 2)
    if k < 1:
        return None, 0.0, "no_endpoint_window"
    start = q[["X", "Y"]].iloc[:k].median().to_numpy(float)
    end = q[["X", "Y"]].iloc[-k:].median().to_numpy(float)
    vec = end - start
    length = float(np.linalg.norm(vec))
    if not np.isfinite(length) or length < AZIMUTH_MIN_DISPLACEMENT:
        return None, length if np.isfinite(length) else 0.0, "short_displacement"
    return vec / length, length, "ok"

def _fit_azimuth_axis(train_wids, data_dir):
    """Fit an unsigned dominant axis on training-well X/Y only."""
    matrix = np.zeros((2, 2), dtype=float)
    valid = 0
    audit = []
    for wid in sorted(train_wids):
        path = Path(data_dir) / f"{wid}__horizontal_well.csv"
        hw = pd.read_csv(path, usecols=["X", "Y", "MD"])
        unit, length, status = _trajectory_unit_vector(hw)
        audit.append({"well": wid, "length": length, "status": status})
        if unit is not None:
            matrix += np.outer(unit, unit)
            valid += 1
    if valid == 0:
        raise RuntimeError("No valid training trajectories available to fit azimuth axis")
    values, vectors = np.linalg.eigh(matrix)
    axis = vectors[:, int(np.argmax(values))].astype(float)
    axis /= np.linalg.norm(axis)
    if axis[0] < 0 or (abs(axis[0]) < 1e-8 and axis[1] > 0):
        axis = -axis
    return axis, pd.DataFrame(audit)

def _azimuth_features(hw, axis):
    unit, length, status = _trajectory_unit_vector(hw)
    if unit is None or axis is None:
        return {"az_dir": 0.0, "az_conf": 0.0, "az_cos": 0.0, "az_sin": 0.0,
                "az_status": status, "az_length": float(length)}
    projection = float(np.dot(unit, np.asarray(axis, float)))
    confidence = abs(projection)
    direction = 0.0 if confidence < AZIMUTH_MIN_CONFIDENCE else (1.0 if projection >= 0 else -1.0)
    return {"az_dir": direction, "az_conf": confidence,
            "az_cos": float(unit[0]), "az_sin": float(unit[1]),
            "az_status": "ambiguous_cross_axis" if direction == 0 else "ok",
            "az_length": float(length)}


In [ ]:
def build_well(hw_path, tw_path, is_train, likpf_map=None):
    global _FI, _DI
    wid = Path(hw_path).stem.replace("__horizontal_well", "")
    try: hw = pd.read_csv(hw_path); tw = pd.read_csv(tw_path).sort_values("TVT")
    except: return None
    if is_train and "TVT" not in hw.columns: return None
    kn = hw[hw.TVT_input.notna()]; ev = hw[hw.TVT_input.isna()]
    if len(ev) == 0 or len(kn) < 10: return None
    if is_train and hw.TVT.isna().all(): return None
    tw_tvt = tw.TVT.to_numpy(np.float32); tw_gr = tw.GR.to_numpy(np.float32)
    if len(tw_tvt) < 3: return None
    pf_a, std_a = run_pf_ancc(hw, tw_tvt, tw_gr)
    if len(pf_a) == 0: return None
    pf_z, std_z = run_pf_z(hw, tw_tvt, tw_gr)
    pf_use = pf_a.astype(np.float32); std_use = std_a.astype(np.float32)
    has_z = len(pf_z) == len(pf_a) and not np.any(np.isnan(pf_z))
    lk = kn.iloc[-1]; last_tvt = float(lk.TVT_input)
    gr_full = hw.GR.astype(float).interpolate(limit_direction="both").fillna(float(np.nanmean(tw_gr)))
    hgr = gr_full.iloc[ev.index[0]:].to_numpy(np.float32); kgr = gr_full.iloc[:len(kn)].to_numpy(np.float32)
    bpaths = {tag: beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs, mc, es, r) for (bs, mc, es, r, tag) in BEAMS}
    beam_ref = (bpaths["cons"]+bpaths["sm5"])/2.
    ktvt = kn.TVT_input.to_numpy(np.float32)
    sc_res, sc_ens = multi_scale_ncc(kgr, ktvt, hgr, hws=(8, 15, 25), stride=3)
    sc8, sc8s = sc_res[0]; sc15, sc15s = sc_res[1]; sc25, sc25s = sc_res[2]; sc_cons = (sc8+sc15+sc25)/3.
    sc_trust = float(np.clip(len(kn)/200., 0., 0.6)); hyb_ref = (1-sc_trust)*beam_ref+sc_trust*sc_ens
    tw_at_k = np.interp(ktvt, tw_tvt, tw_gr).astype(np.float32); a_cal, b_cal = affine_cal(kgr, tw_at_k)
    kmd = kn.MD.to_numpy(np.float32); kz = kn.Z.to_numpy(np.float32)
    pfx_rmse = float(np.sqrt(np.mean((kgr-tw_at_k)**2)))
    slp_all = robust_slope(kmd, ktvt); slp_50 = robust_slope(kmd[-50:], ktvt[-50:]); slp_z = robust_slope(kz, ktvt)
    swid = wid if is_train else None
    xy_ev = ev[["X","Y"]].to_numpy(np.float64); xy_kn = kn[["X","Y"]].to_numpy(np.float64)
    form_ev, knn_d = _FI.impute(xy_ev, self_wid=swid); form_kn, _ = _FI.impute(xy_kn, self_wid=swid)
    z_kn = kn.Z.to_numpy(np.float32); z_ev = ev.Z.to_numpy(np.float32)
    tvt_fs = {}; form_rmse = {}; form_list = []
    for fi2, fn in enumerate(FORMATIONS):
        b_full, b_early, b_mid, b_late, b_wls = seg_b_well(ktvt, z_kn, form_kn[:, fi2])
        tvt_f = (-z_ev+form_ev[:, fi2]+b_full).astype(np.float32)
        tvt_fs[f"tvtF_{fn}"]=tvt_f; tvt_fs[f"tvtFw_{fn}"]=(-z_ev+form_ev[:,fi2]+b_wls).astype(np.float32)
        tvt_fs[f"tvtF50_{fn}"]=(-z_ev+form_ev[:,fi2]+b_late).astype(np.float32)
        tvt_fs[f"bw_{fn}"]=np.float32(b_full); tvt_fs[f"bww_{fn}"]=np.float32(b_wls); tvt_fs[f"bw50_{fn}"]=np.float32(b_late)
        tvt_fs[f"bw_early_{fn}"]=np.float32(b_early); tvt_fs[f"bw_mid_{fn}"]=np.float32(b_mid)
        form_rmse[fn]=float(np.sqrt(np.mean((ktvt-(-z_kn+form_kn[:,fi2]+b_full))**2))); form_list.append(tvt_f)
    fs = np.stack(form_list, 1)
    form_mean_d=(fs.mean(1)-last_tvt).astype(np.float32); form_std_d=fs.std(1).astype(np.float32); form_rng_d=(fs.max(1)-fs.min(1)).astype(np.float32)
    d_ancc, d_std, d_dist = _DI.impute(xy_ev, self_wid=swid); d_kn, d_std_kn, _ = _DI.impute(xy_kn, self_wid=swid)
    _, b_de, b_dm, b_dl, b_dw = seg_b_well(ktvt, z_kn, d_kn); b_d = float(np.median(ktvt+z_kn-d_kn))
    tvt_dense=(-z_ev+d_ancc+b_d).astype(np.float32); tvt_densew=(-z_ev+d_ancc+b_dw).astype(np.float32); tvt_dense50=(-z_ev+d_ancc+b_dl).astype(np.float32)
    res_kn = ktvt+z_kn-d_kn; d_rmse=float(np.sqrt(np.mean(res_kn**2))); d_bias=float(np.mean(res_kn)); d_nb_std=float(np.mean(d_std_kn))
    all_sigs=[pf_use]+list(bpaths.values())+[sc8,sc15,sc25,sc_ens,tvt_fs["tvtF_ANCC"],tvt_dense]
    sig_mat=np.stack(all_sigs,1); sig_std=sig_mat.std(1).astype(np.float32); sig_mean=(sig_mat.mean(1)-last_tvt).astype(np.float32)
    gr_s=pd.Series(gr_full.values); rolls={}
    for w in [5,21,51,101]:
        r=gr_s.rolling(w,center=True,min_periods=1); rolls[f"grm{w}"]=r.mean().iloc[ev.index].values.astype(np.float32); rolls[f"grs{w}"]=r.std().fillna(0).iloc[ev.index].values.astype(np.float32)
    for lag in [1,5,15,30]:
        rolls[f"glag{lag}"]=gr_s.shift(lag).bfill().iloc[ev.index].values.astype(np.float32); rolls[f"glead{lag}"]=gr_s.shift(-lag).ffill().iloc[ev.index].values.astype(np.float32)
    gr_d1=gr_s.diff().fillna(0.).iloc[ev.index].values.astype(np.float32); gr_d2=gr_s.diff().diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
    gr_env=gr_s.rolling(21,center=True,min_periods=1).max().iloc[ev.index].values.astype(np.float32)
    gr_nrg=np.sqrt(np.maximum((gr_s**2).rolling(21,center=True,min_periods=1).mean(),0.)).iloc[ev.index].values.astype(np.float32)
    hmd=ev.MD.to_numpy(np.float32); md_since=hmd-float(lk.MD)
    slp_b_all=(last_tvt+slp_all*md_since).astype(np.float32); slp_b_50=(last_tvt+slp_50*md_since).astype(np.float32)
    mdd=hw.MD.diff().replace(0,np.nan)
    dzdmd=(hw.Z.diff()/mdd).iloc[ev.index].values.astype(np.float32); dxdmd=(hw.X.diff()/mdd).iloc[ev.index].values.astype(np.float32); dydmd=(hw.Y.diff()/mdd).iloc[ev.index].values.astype(np.float32)
    nh=len(ev); frac=(np.arange(nh)/max(nh-1,1)).astype(np.float32)
    def sc(v): return np.full(nh, np.float32(v), np.float32)
    feats={"well":wid,"id":[f"{wid}_{i}" for i in ev.index],"last_known_tvt":sc(last_tvt),
        "pf_ancc":pf_use,"pf_ancc_std":std_use,"pf_ancc_delta":(pf_use-last_tvt).astype(np.float32),
        "pf_z":(pf_z.astype(np.float32) if has_z else sc(last_tvt)),"pf_z_delta":((pf_z-last_tvt).astype(np.float32) if has_z else sc(0.)),
        "pf_vs_z":((pf_use-pf_z.astype(np.float32)) if has_z else sc(0.)),
        **{f"beam_{t}_d":(p-np.float32(last_tvt)).astype(np.float32) for t,p in bpaths.items()},
        "beam_mean_d":np.stack([(p-last_tvt) for p in bpaths.values()],1).mean(1).astype(np.float32),
        "beam_std_d":np.stack([(p-last_tvt) for p in bpaths.values()],1).std(1).astype(np.float32),
        "beam_med_d":np.median(np.stack([(p-last_tvt) for p in bpaths.values()],1),1).astype(np.float32),
        "sc8_d":(sc8-np.float32(last_tvt)).astype(np.float32),"sc8_sc":sc8s,"sc15_d":(sc15-np.float32(last_tvt)).astype(np.float32),"sc15_sc":sc15s,
        "sc25_d":(sc25-np.float32(last_tvt)).astype(np.float32),"sc25_sc":sc25s,"sc_cons_d":(sc_cons-np.float32(last_tvt)).astype(np.float32),
        "sc_ens_d":(sc_ens-np.float32(last_tvt)).astype(np.float32),"sc_trust":sc(sc_trust),"hyb_d":(hyb_ref-np.float32(last_tvt)).astype(np.float32),
        "sig_std":sig_std,"sig_mean_d":sig_mean,**tvt_fs,**{f"frm_rmse_{fn}":sc(form_rmse[fn]) for fn in FORMATIONS},
        "form_mean_d":form_mean_d,"form_std_d":form_std_d,"form_rng_d":form_rng_d,
        "spatial_ancc_d":(form_ev[:,0]-np.float32(np.interp(last_tvt,tw_tvt,tw_gr))),"spatial_knn_dist":knn_d,
        "dense_ancc":d_ancc,"dense_std":d_std,"dense_dist":d_dist,"tvt_dense_d":(tvt_dense-last_tvt).astype(np.float32),
        "tvt_densew_d":(tvt_densew-last_tvt).astype(np.float32),"tvt_dense50_d":(tvt_dense50-last_tvt).astype(np.float32),
        "dense_rmse":sc(d_rmse),"dense_bias":sc(d_bias),"dense_nb_std":sc(d_nb_std),
        "pf_vs_spatial":(pf_use-tvt_fs["tvtF_ANCC"]).astype(np.float32),"pf_vs_dense":(pf_use-tvt_dense).astype(np.float32),
        "spatial_vs_dense":(tvt_fs["tvtF_ANCC"]-tvt_dense).astype(np.float32),"beam_vs_spatial":(bpaths["cons"]-tvt_fs["tvtF_ANCC"]).astype(np.float32),
        "sc_vs_beam":(sc_ens-bpaths["cons"]).astype(np.float32),"cal_a":sc(a_cal),"cal_b":sc(b_cal),
        "pfx_rmse":sc(pfx_rmse),"known_len":sc(len(kn)),"eval_len":sc(nh),"slp_all":sc(slp_all),"slp_50":sc(slp_50),"slp_z":sc(slp_z),
        "slp_b_d_all":(slp_b_all-last_tvt).astype(np.float32),"slp_b_d_50":(slp_b_50-last_tvt).astype(np.float32),
        "ktvt_range":sc(float(np.ptp(ktvt))),"ktvt_std":sc(float(ktvt.std())),"md_since":md_since,"frac":frac,"frac2":frac**2,"sqrt_frac":np.sqrt(frac),
        "z":z_ev,"dx":(ev.X-float(lk.X)).to_numpy(np.float32),"dy":(ev.Y-float(lk.Y)).to_numpy(np.float32),"dz":(z_ev-float(lk.Z)).astype(np.float32),
        "dxy":np.sqrt((ev.X-float(lk.X))**2+(ev.Y-float(lk.Y))**2).to_numpy(np.float32),"dzdmd":dzdmd,"dxdmd":dxdmd,"dydmd":dydmd,
        "gr":hgr,"gr_d1":gr_d1,"gr_d2":gr_d2,"gr_env":gr_env,"gr_nrg":gr_nrg,
        "gr_vs_tw_anc":hgr-np.float32(np.interp(last_tvt,tw_tvt,tw_gr)),"gr_vs_slp_all":hgr-np.interp(slp_b_all,tw_tvt,tw_gr).astype(np.float32),
        **{f"tda{int(o)}":hgr-np.float32(np.interp(last_tvt+o,tw_tvt,tw_gr)) for o in ANCH_OFFS},
        **{f"tdbc{int(o)}":hgr-np.interp(beam_ref+o,tw_tvt,tw_gr).astype(np.float32) for o in BEAM_OFFS},
        **{f"tdsc{int(o)}":hgr-np.interp(sc_ens+o,tw_tvt,tw_gr).astype(np.float32) for o in SC_OFFS},
        **{f"tdpf{int(o)}":hgr-np.interp(pf_use+o,tw_tvt,tw_gr).astype(np.float32) for o in PF_OFFS},
        "tw_range":sc(float(np.ptp(tw_tvt))),"tw_gr_mean":sc(float(tw_gr.mean()))}
    az = _azimuth_features(hw, _AZ_AXIS)
    feats.update({name: sc(az[name]) for name in AZIMUTH_FEATURES})
    for k,v in rolls.items(): feats[k]=v
    res = pd.DataFrame(feats)
    if is_train: res["target"]=(ev.TVT.to_numpy(np.float32)-np.float32(last_tvt))
    return res

def init_imputers(train_wids):
    global _FI, _DI, _AZ_AXIS, _AZ_AXIS_AUDIT
    _AZ_AXIS, _AZ_AXIS_AUDIT = _fit_azimuth_axis(train_wids, CFG.DATA/"train")
    _FI = FormationPlaneKNN(train_wids, CFG.DATA/"train"); _DI = DenseANCCImputer(train_wids, CFG.DATA/"train")

def _likpf_rows(wid, split):
    hw, tw = load_well(wid, split)
    out, idx, _ = lik_pf(hw, tw)
    if not len(out): return None
    d = {"id": [f"{wid}_{i}" for i in idx]}
    for k, v in out.items():
        d["likpf_" + k.replace("pf_scale_", "scale_").replace("pf_mean", "mean")] = v.astype(np.float32)
    return pd.DataFrame(d)

def build_likpf(wids, split):
    # threads are safe here: the lik-PF numba kernel is compiled with nogil=True, so it
    # releases the GIL and parallelises across threads (no pickling of numba code needed).
    res = Parallel(n_jobs=CFG.n_jobs, prefer="threads")(delayed(_likpf_rows)(w, split) for w in wids)
    return pd.concat([r for r in res if r is not None], ignore_index=True)

def build_features(wids, split, is_train):
    paths = [CFG.DATA/split/f"{w}__horizontal_well.csv" for w in wids]
    res = Parallel(n_jobs=CFG.n_jobs, prefer="threads")(
        delayed(build_well)(str(p), str(p.parent/f"{p.stem.replace('__horizontal_well','')}__typewell.csv"), is_train)
        for p in paths if (p.parent/f"{p.stem.replace('__horizontal_well','')}__typewell.csv").exists())
    parts = [r for r in res if r is not None]
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

def add_likpf_features(df, likpf):
    df = df.merge(likpf, on="id", how="left")
    for c in [c for c in likpf.columns if c != "id"]:
        df[c] = df[c].fillna(df["last_known_tvt"]); df[c+"_d"] = (df[c]-df["last_known_tvt"]).astype(np.float32)
    return df


In [ ]:
def _device():
    if CFG.USE_GPU == "cpu": return "cpu", "CPU"
    if CFG.USE_GPU == "gpu": return "gpu", "GPU"
    try:  # detect a real NVIDIA GPU (Kaggle GPU accelerator) via nvidia-smi
        import subprocess
        if subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0:
            return "gpu", "GPU"
    except Exception:
        pass
    return "cpu", "CPU"

def lgb_configs(dev):
    base = dict(boosting_type="gbdt", objective="regression", verbose=-1, n_jobs=-1, max_bin=255)
    if dev == "gpu": base.update(device_type="gpu", gpu_use_dp=False)
    n = 600 if CFG.FAST else 5000
    return [
        dict(**base, num_leaves=255, min_child_samples=15, subsample=0.8, subsample_freq=1,
             colsample_bytree=0.8, reg_lambda=3.0, reg_alpha=0.05, learning_rate=0.03, n_estimators=n, seed=123),
        dict(**base, num_leaves=64, min_child_samples=40, subsample=0.474, subsample_freq=1,
             colsample_bytree=0.393, reg_lambda=95.75, reg_alpha=10.79, min_child_weight=0.24,
             learning_rate=0.0093, n_estimators=min(2*n, 10000), random_state=0),
        dict(**base, num_leaves=64, min_child_samples=40, subsample=0.474, subsample_freq=1,
             colsample_bytree=0.393, reg_lambda=95.75, reg_alpha=10.79, min_child_weight=0.24,
             learning_rate=0.0093, n_estimators=min(2*n, 10000), random_state=29),
    ]

def cb_configs(dev):
    tt = "GPU" if dev == "gpu" else "CPU"
    n = 800 if CFG.FAST else 8000
    return [
        dict(iterations=n, depth=7, l2_leaf_reg=2.0, min_data_in_leaf=15, border_count=254,
             loss_function="RMSE", task_type=tt, od_type="Iter", od_wait=300, verbose=0, learning_rate=0.02, random_seed=7),
        dict(iterations=n, depth=7, l2_leaf_reg=2.0, min_data_in_leaf=15, border_count=254,
             loss_function="RMSE", task_type=tt, od_type="Iter", od_wait=300, verbose=0, learning_rate=0.03, random_seed=123),
    ]

def train_stack(train_df, test_df, features):
    from lightgbm import LGBMRegressor, early_stopping, log_evaluation
    from catboost import CatBoostRegressor
    from sklearn.model_selection import GroupKFold
    from sklearn.linear_model import Ridge
    dev, devname = _device(); print("device:", devname)
    X = train_df[features].values.astype(np.float32); y = train_df["target"].values.astype(np.float32)
    g = train_df["well"].values; Xt = test_df[features].values.astype(np.float32)
    cv = GroupKFold(CFG.n_splits); oof_cols = {}; test_cols = {}
    def run(name, make, fit_kw, is_lgb):
        # LightGBM: slice to best_iteration_ via num_iteration. CatBoost: use_best_model
        # already trims to the best tree, and its predict() takes no num_iteration kwarg.
        oof = np.zeros(len(train_df)); tp = np.zeros(len(test_df))
        for tr, va in cv.split(X, y, groups=g):
            m = make(); m.fit(X[tr], y[tr], eval_set=[(X[va], y[va])], **fit_kw)
            if is_lgb:
                it = m.best_iteration_
                oof[va] = m.predict(X[va], num_iteration=it); tp += m.predict(Xt, num_iteration=it) / CFG.n_splits
            else:
                oof[va] = m.predict(X[va]); tp += m.predict(Xt) / CFG.n_splits
        oof_cols[name] = oof; test_cols[name] = tp
        print(f"  {name}: OOF RMSE={rmse(y, oof):.4f}", flush=True)
    for i, p in enumerate(lgb_configs(dev)):
        run(f"lgb{i}", lambda p=p: LGBMRegressor(**p),
            dict(eval_metric="rmse", callbacks=[early_stopping(250, verbose=False), log_evaluation(0)]), True)
    for i, p in enumerate(cb_configs(dev)):
        run(f"cb{i}", lambda p=p: CatBoostRegressor(**p),
            dict(early_stopping_rounds=250, use_best_model=True), False)
    OOF = pd.DataFrame(oof_cols); TEST = pd.DataFrame(test_cols)
    rid = Ridge(alpha=1.66, positive=True, fit_intercept=True); meta = np.zeros(len(train_df))
    for tr, va in cv.split(OOF.values, y, groups=g):
        rid.fit(OOF.values[tr], y[tr]); meta[va] = rid.predict(OOF.values[va])
    rid.fit(OOF.values, y); meta_test = rid.predict(TEST.values)
    print(f"  ridge-stack OOF RMSE={rmse(y, meta):.4f}")
    return meta, meta_test, OOF, TEST


In [ ]:
class PP:   # tuned on 773-well GroupKFold OOF (Nelder-Mead + grid; the optimum is flat)
    alpha = 1.0         # global scale on the learned delta (tuned ~1.0)
    tau = 85.0          # warm-up length in ft: damps the first feet after PS (tuned ~90)
    w_pf = 0.0          # blending the model with the single PF no longer helps once lik-PF is a feature
    w_sub1 = 0.60       # weight on the learned model; lik-PF gets 1-w_sub1. CV optimum ~0.68 (flat
                        # 0.55-0.68); 0.60 is a small hedge toward the drift-robust lik-PF for LB transfer.
    sub2_scale = "scale_5"   # which likelihood-scale of the lik-PF to use as sub2 (3/5/8 ~equivalent)
    sg_win = 61         # per-well Savitzky-Golay smoothing window (effect is small, ~0.01 ft)
    sg_poly = 3

def warmup(md_since, tau): return 1.-np.exp(-np.maximum(md_since, 0.)/tau) if tau > 1e-6 else 1.0

def make_prediction(df, model_delta, likpf):
    last = df["last_known_tvt"].values.astype(float)
    pf_delta = df["pf_ancc"].values.astype(float) - last
    lp = df[f"likpf_{PP.sub2_scale}"].values.astype(float) - last
    sub1 = PP.alpha*warmup(df["md_since"].values.astype(float), PP.tau)*(model_delta*(1-PP.w_pf)+pf_delta*PP.w_pf)
    delta = PP.w_sub1*sub1 + (1-PP.w_sub1)*lp
    pred = last + delta
    # per-well Savitzky-Golay smoothing
    out = pred.copy(); dfx = df.reset_index(drop=True)
    for _, idx in dfx.groupby("well", sort=False).groups.items():
        pos = dfx.index.get_indexer(idx); v = pred[pos]; n = len(v); wl = min(PP.sg_win, n)
        if wl % 2 == 0: wl -= 1
        if wl >= PP.sg_poly+2: out[pos] = savgol_filter(v, wl, PP.sg_poly)
    return out


## Paired validation and packaging

The augmented frame is built once because PF-derived features include stochastic components. Control and conditioned matrices are sliced from those exact rows, and one stored GroupKFold split list is reused for every model.

In [ ]:
# ---- paired training, reporting, and compatible packaging ----------------------
def _find_baseline_feature_contract():
    candidates = []
    for root in BASELINE_MODEL_ROOTS:
        p = Path(root)
        if p.exists():
            candidates.extend([p, *[q.parent for q in p.rglob("features.json")]])
    seen = set()
    for directory in candidates:
        directory = Path(directory)
        if directory in seen:
            continue
        seen.add(directory)
        fp = directory / "features.json"
        if fp.exists() and all((directory / f"lgb{i}.pkl").exists() for i in range(3)):
            features = json.loads(fp.read_text())
            if isinstance(features, list) and len(features) == 196 and len(set(features)) == 196:
                return directory, features
    raise FileNotFoundError("Could not locate the public 196-feature + lgb0/1/2 artifact contract")

def _fixed_group_folds(frame):
    from sklearn.model_selection import GroupKFold
    groups = frame["well"].astype(str).to_numpy()
    dummy = np.zeros(len(frame), dtype=np.float32)
    folds = [(tr.copy(), va.copy()) for tr, va in GroupKFold(n_splits=5).split(dummy, dummy, groups)]
    coverage = np.zeros(len(frame), dtype=np.int8)
    for _, va in folds:
        coverage[va] += 1
    assert np.all(coverage == 1)
    return folds

def _fit_paired_oof(frame, control_features, conditioned_features, folds):
    from lightgbm import LGBMRegressor, early_stopping, log_evaluation
    y = frame["target"].to_numpy(np.float32)
    matrices = {
        "control": frame[control_features].to_numpy(np.float32),
        "conditioned": frame[conditioned_features].to_numpy(np.float32),
    }
    configs = lgb_configs(_device()[0])
    predictions = {name: np.zeros((len(frame), len(configs)), np.float64) for name in matrices}
    fold_models = {name: [[] for _ in configs] for name in matrices}
    best_iterations = {name: [[] for _ in configs] for name in matrices}
    for variant, X in matrices.items():
        print(f"paired OOF: {variant} ({X.shape[1]} features)", flush=True)
        for config_idx, params in enumerate(configs):
            for fold_idx, (tr, va) in enumerate(folds):
                model = LGBMRegressor(**params)
                model.fit(
                    X[tr], y[tr], eval_set=[(X[va], y[va])], eval_metric="rmse",
                    callbacks=[early_stopping(250, verbose=False), log_evaluation(0)],
                )
                iteration = int(model.best_iteration_ or params["n_estimators"])
                predictions[variant][va, config_idx] = model.predict(X[va], num_iteration=iteration)
                fold_models[variant][config_idx].append(model)
                best_iterations[variant][config_idx].append(iteration)
                print(f"  {variant} lgb{config_idx} fold{fold_idx}: best_iteration={iteration}", flush=True)
    ensemble = {name: values.mean(axis=1) for name, values in predictions.items()}
    return ensemble, fold_models, best_iterations, configs

def _prediction_levels(frame, delta):
    learned = frame["last_known_tvt"].to_numpy(float) + np.asarray(delta, float)
    learned_pf = make_prediction(frame, np.asarray(delta, float), None)
    # The exact SP45 projection is test-only in the clean notebook. pf_ancc is the
    # fixed local anchor proxy, shared byte-for-byte between paired candidates.
    sp45_proxy = frame["pf_ancc"].to_numpy(float)
    final_proxy = 0.60 * sp45_proxy + 0.40 * learned_pf
    return {"learned": learned, "make_prediction": learned_pf, "final_proxy": final_proxy}

def _rmse_truth(truth, pred):
    truth = np.asarray(truth, float); pred = np.asarray(pred, float)
    return float(np.sqrt(np.mean((truth - pred) ** 2)))

def _paired_metric_rows(frame, control_levels, conditioned_levels, fold_ids=None, prefix_fraction=None):
    truth = frame["last_known_tvt"].to_numpy(float) + frame["target"].to_numpy(float)
    direction = frame["az_dir"].to_numpy(float)
    records = []
    slices = [("pooled", np.ones(len(frame), bool))]
    slices += [(f"az_dir_{int(group):+d}", direction == group) for group in (-1.0, 0.0, 1.0)]
    if fold_ids is not None:
        slices += [(f"fold_{fold}", np.asarray(fold_ids) == fold) for fold in sorted(set(fold_ids))]
    for level in control_levels:
        for subset, mask in slices:
            if not np.any(mask):
                continue
            c = _rmse_truth(truth[mask], control_levels[level][mask])
            a = _rmse_truth(truth[mask], conditioned_levels[level][mask])
            records.append({"prefix_fraction": prefix_fraction, "level": level, "subset": subset, "aggregation": "row_weighted",
                            "rows": int(mask.sum()), "control_rmse": c, "conditioned_rmse": a,
                            "improvement_control_minus_conditioned": c - a})
            positions = np.flatnonzero(mask)
            wells = frame.iloc[positions]["well"].astype(str).to_numpy()
            well_scores = []
            for wid in np.unique(wells):
                wm = wells == wid
                well_scores.append((_rmse_truth(truth[positions][wm], control_levels[level][positions][wm]),
                                    _rmse_truth(truth[positions][wm], conditioned_levels[level][positions][wm])))
            c_equal = float(np.mean([x[0] for x in well_scores]))
            a_equal = float(np.mean([x[1] for x in well_scores]))
            records.append({"prefix_fraction": prefix_fraction, "level": level, "subset": subset, "aggregation": "equal_well",
                            "rows": int(mask.sum()), "wells": len(well_scores), "control_rmse": c_equal,
                            "conditioned_rmse": a_equal, "improvement_control_minus_conditioned": c_equal - a_equal})
    return records

def _well_bootstrap_delta(frame, control_pred, conditioned_pred, draws=WELL_BOOTSTRAP_DRAWS):
    truth = frame["last_known_tvt"].to_numpy(float) + frame["target"].to_numpy(float)
    tmp = pd.DataFrame({"well": frame["well"].astype(str),
                        "control_sq": (truth - np.asarray(control_pred)) ** 2,
                        "conditioned_sq": (truth - np.asarray(conditioned_pred)) ** 2})
    by_well = tmp.groupby("well", sort=True).agg(control_sum=("control_sq", "sum"),
                                                  conditioned_sum=("conditioned_sq", "sum"),
                                                  rows=("well", "size")).reset_index()
    rng = np.random.default_rng(BUILDER_SEED)
    values = np.empty(draws, dtype=float)
    n = len(by_well)
    for i in range(draws):
        pick = rng.integers(0, n, size=n)
        rows = by_well["rows"].to_numpy()[pick].sum()
        c = np.sqrt(by_well["control_sum"].to_numpy()[pick].sum() / rows)
        a = np.sqrt(by_well["conditioned_sum"].to_numpy()[pick].sum() / rows)
        values[i] = c - a
    return {"draws": int(draws), "mean": float(values.mean()),
            "ci_low": float(np.quantile(values, 0.025)), "ci_high": float(np.quantile(values, 0.975))}

def _direction_balanced_mask_wells(train_wids):
    groups = {-1: [], 0: [], 1: []}
    for wid in sorted(train_wids):
        hw = pd.read_csv(CFG.DATA / "train" / f"{wid}__horizontal_well.csv")
        if int(hw["TVT_input"].notna().sum()) < MASKED_PREFIX_MIN_KNOWN:
            continue
        groups[int(_azimuth_features(hw, _AZ_AXIS)["az_dir"])].append(wid)
    if MASKED_PREFIX_MAX_WELLS <= 0:
        selected = sorted(sum(groups.values(), []))
    else:
        quota = max(1, MASKED_PREFIX_MAX_WELLS // 3)
        selected = sorted(sum((values[:quota] for values in groups.values()), []))
        if len(selected) < MASKED_PREFIX_MAX_WELLS:
            remainder = [w for w in sorted(sum(groups.values(), [])) if w not in set(selected)]
            selected.extend(remainder[:MASKED_PREFIX_MAX_WELLS - len(selected)])
    return selected, {str(k): len(v) for k, v in groups.items()}

def _masked_feature_frame(wid, fraction):
    hw, tw = load_well(wid, "train")
    known_idx = hw.index[hw["TVT_input"].notna()].to_numpy()
    keep_n = max(10, min(len(known_idx) - 1, int(np.floor(len(known_idx) * fraction))))
    masked = hw.copy()
    masked.loc[known_idx[keep_n:], "TVT_input"] = np.nan
    with tempfile.TemporaryDirectory(prefix="rogii_mask_") as tmp:
        hp = Path(tmp) / f"{wid}__horizontal_well.csv"
        tp = Path(tmp) / f"{wid}__typewell.csv"
        masked.to_csv(hp, index=False); tw.to_csv(tp, index=False)
        feature_frame = build_well(str(hp), str(tp), is_train=True)
    if feature_frame is None or feature_frame.empty:
        return None
    out, idx, _ = lik_pf(masked, tw)
    lik = {"id": [f"{wid}_{i}" for i in idx]}
    for name, values in out.items():
        lik["likpf_" + name.replace("pf_scale_", "scale_").replace("pf_mean", "mean")] = values.astype(np.float32)
    return add_likpf_features(feature_frame, pd.DataFrame(lik)).reset_index(drop=True)

def _run_masked_prefix(train_wids, control_features, conditioned_features, fold_models, well_to_fold):
    selected, eligible_counts = _direction_balanced_mask_wells(train_wids)
    rows = []
    summaries = []
    for fraction in MASKED_PREFIX_FRACTIONS:
        print(f"masked-prefix fraction {fraction:.2f}: {len(selected)} wells", flush=True)
        pieces = []
        control_delta = []; conditioned_delta = []
        for wid in selected:
            frame = _masked_feature_frame(wid, fraction)
            if frame is None or frame.empty or wid not in well_to_fold:
                continue
            fold = well_to_fold[wid]
            matrices = {
                "control": frame[control_features].to_numpy(np.float32),
                "conditioned": frame[conditioned_features].to_numpy(np.float32),
            }
            pred = {}
            for variant, X in matrices.items():
                pred[variant] = np.mean(
                    [fold_models[variant][config_idx][fold].predict(X) for config_idx in range(3)], axis=0)
            pieces.append(frame)
            control_delta.append(pred["control"]); conditioned_delta.append(pred["conditioned"])
        if not pieces:
            continue
        frame = pd.concat(pieces, ignore_index=True)
        cdelta = np.concatenate(control_delta); adelta = np.concatenate(conditioned_delta)
        clevels = _prediction_levels(frame, cdelta); alevels = _prediction_levels(frame, adelta)
        summaries.extend(_paired_metric_rows(frame, clevels, alevels, prefix_fraction=fraction))
        truth = frame["last_known_tvt"].to_numpy(float) + frame["target"].to_numpy(float)
        detail = frame[["well", "id", "az_dir"]].copy()
        detail["prefix_fraction"] = fraction; detail["truth_tvt"] = truth
        for level in clevels:
            detail[f"control_{level}"] = clevels[level]
            detail[f"conditioned_{level}"] = alevels[level]
        rows.append(detail)
    detail = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    return detail, pd.DataFrame(summaries), selected, eligible_counts

def _fit_and_save_full_models(frame, features, configs, best_iterations, out_dir):
    import joblib
    from lightgbm import LGBMRegressor
    out_dir.mkdir(parents=True, exist_ok=True)
    X = frame[features].to_numpy(np.float32); y = frame["target"].to_numpy(np.float32)
    for i, params in enumerate(configs):
        full_params = dict(params)
        full_params["n_estimators"] = int(np.median(best_iterations[i]))
        model = LGBMRegressor(**full_params)
        model.fit(X, y)
        joblib.dump(model, out_dir / f"lgb{i}.pkl")
    (out_dir / "features.json").write_text(json.dumps(features, indent=2) + "\n")

def build_paired_artifacts():
    import joblib, lightgbm, sklearn
    started = time.time()
    baseline_dir, control_features = _find_baseline_feature_contract()
    assert len(control_features) == 196 and not set(AZIMUTH_FEATURES).intersection(control_features)
    conditioned_features = list(control_features) + list(AZIMUTH_FEATURES)
    assert conditioned_features[:196] == control_features
    assert conditioned_features[-4:] == list(AZIMUTH_FEATURES)
    assert len(conditioned_features) == 200 and len(set(conditioned_features)) == 200

    train_wids = sorted(p.stem.replace("__horizontal_well", "") for p in (CFG.DATA / "train").glob("*__horizontal_well.csv"))
    if CFG.N_TRAIN_WELLS:
        train_wids = train_wids[:CFG.N_TRAIN_WELLS]
    if len(train_wids) < CFG.n_splits:
        raise RuntimeError(f"Need at least {CFG.n_splits} train wells, found {len(train_wids)}")
    init_imputers(train_wids)
    print("azimuth axis:", _AZ_AXIS.tolist(), flush=True)

    # Critical isolation rule: build stochastic PF-derived rows exactly once, then
    # slice the same frame into control and conditioned matrices.
    likpf_train = build_likpf(train_wids, "train")
    augmented = add_likpf_features(build_features(train_wids, "train", is_train=True), likpf_train).reset_index(drop=True)
    missing = [c for c in conditioned_features if c not in augmented.columns]
    if missing:
        raise KeyError(f"Missing required deployed features: {missing}")
    if not np.isfinite(augmented["target"].to_numpy(float)).all():
        raise ValueError("Non-finite training targets in paired feature frame")
    if not np.isfinite(augmented[list(AZIMUTH_FEATURES)].to_numpy(float)).all():
        raise ValueError("Non-finite azimuth features in paired feature frame")

    folds = _fixed_group_folds(augmented)
    fold_ids = np.full(len(augmented), -1, dtype=int)
    well_to_fold = {}
    for fold, (_, va) in enumerate(folds):
        fold_ids[va] = fold
        for wid in augmented.iloc[va]["well"].astype(str).unique():
            if wid in well_to_fold and well_to_fold[wid] != fold:
                raise AssertionError("A well crossed GroupKFold folds")
            well_to_fold[wid] = fold

    oof, fold_models, best_iterations, configs = _fit_paired_oof(
        augmented, control_features, conditioned_features, folds)
    control_levels = _prediction_levels(augmented, oof["control"])
    conditioned_levels = _prediction_levels(augmented, oof["conditioned"])
    primary_metrics = pd.DataFrame(_paired_metric_rows(
        augmented, control_levels, conditioned_levels, fold_ids=fold_ids))
    bootstrap = _well_bootstrap_delta(
        augmented, control_levels["final_proxy"], conditioned_levels["final_proxy"])

    oof_detail = augmented[["well", "id", "az_dir", "az_conf", "target", "last_known_tvt"]].copy()
    oof_detail["fold"] = fold_ids
    for level in control_levels:
        oof_detail[f"control_{level}"] = control_levels[level]
        oof_detail[f"conditioned_{level}"] = conditioned_levels[level]

    masked_detail, masked_metrics, masked_wells, masked_eligible_counts = _run_masked_prefix(
        train_wids, control_features, conditioned_features, fold_models, well_to_fold)

    group_counts = (augmented[["well", "az_dir"]].drop_duplicates()
                    .groupby("az_dir")["well"].nunique().to_dict())
    def metric(table, level, subset, fraction=None, aggregation="row_weighted"):
        q = table[(table.level == level) & (table.subset == subset) & (table.aggregation == aggregation)]
        if fraction is not None:
            q = q[np.isclose(q.prefix_fraction.astype(float), fraction)]
        return None if q.empty else q.iloc[0].to_dict()

    pooled = metric(primary_metrics, "final_proxy", "pooled")
    fold_rows = primary_metrics[(primary_metrics.level == "final_proxy") & primary_metrics.subset.str.startswith("fold_") & (primary_metrics.aggregation == "row_weighted")]
    direction_rows = primary_metrics[(primary_metrics.level == "final_proxy") & primary_metrics.subset.isin(["az_dir_-1", "az_dir_+1"]) & (primary_metrics.aggregation == "row_weighted")]
    masked_pooled = masked_metrics[(masked_metrics.level == "final_proxy") & (masked_metrics.subset == "pooled") & (masked_metrics.aggregation == "row_weighted")]
    gates = {
        "primary_improvement_at_least_0_05": bool(pooled and pooled["improvement_control_minus_conditioned"] >= 0.05),
        "at_least_3_of_5_folds_improve": bool((fold_rows.improvement_control_minus_conditioned > 0).sum() >= 3),
        "neither_direction_regresses_over_0_10": bool(len(direction_rows) == 2 and (direction_rows.improvement_control_minus_conditioned >= -0.10).all()),
        "masked_pooled_not_regress_over_0_05": bool(len(masked_pooled) == 3 and (masked_pooled.improvement_control_minus_conditioned >= -0.05).all()),
        "at_least_2_mask_fractions_improve": bool((masked_pooled.improvement_control_minus_conditioned > 0).sum() >= 2),
        "masked_prefix_all_eligible_run": bool(MASKED_PREFIX_MAX_WELLS == 0),
        "both_direction_groups_at_least_100_wells": bool(group_counts.get(-1.0, 0) >= 100 and group_counts.get(1.0, 0) >= 100),
        "all_predictions_finite": bool(all(np.isfinite(v).all() for v in [*control_levels.values(), *conditioned_levels.values()])),
        "all_oof_rows_covered_once": bool(np.all(fold_ids >= 0)),
        "reduced_cost_screen_only": False,
    }
    deployable = bool(all(gates.values()))

    versions = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
                "lightgbm": lightgbm.__version__, "sklearn": sklearn.__version__, "joblib": joblib.__version__}
    common_manifest = {
        "code_version": ARTIFACT_BUILDER_CODE_VERSION,
        "created_utc": pd.Timestamp.utcnow().isoformat(),
        "source_baseline_artifact": str(baseline_dir),
        "axis": [float(x) for x in _AZ_AXIS],
        "thresholds": {"min_rows": AZIMUTH_MIN_ROWS, "min_displacement": AZIMUTH_MIN_DISPLACEMENT,
                       "min_confidence": AZIMUTH_MIN_CONFIDENCE, "endpoint_fraction": AZIMUTH_ENDPOINT_FRACTION},
        "train_wells": len(train_wids), "train_rows": len(augmented),
        "train_well_group_counts": {str(k): int(v) for k, v in group_counts.items()},
        "folds": 5, "model_semantics": "plain mean of lgb0.pkl, lgb1.pkl, lgb2.pkl on float32 ndarray",
        "masked_prefix": {"fractions": list(MASKED_PREFIX_FRACTIONS), "selected_wells": len(masked_wells),
                          "eligible_group_counts": masked_eligible_counts, "full_eligible_run": MASKED_PREFIX_MAX_WELLS == 0},
        "primary_metrics": primary_metrics.to_dict(orient="records"),
        "masked_prefix_metrics": masked_metrics.to_dict(orient="records"),
        "well_bootstrap_final_proxy": bootstrap,
        "gates": gates, "deployable": deployable,
        "versions": versions,
        "limitations": [
            "final_proxy uses the shared pf_ancc anchor because exact SP45 projection OOF is not available in the clean training block",
            "artifact must be uploaded/attached as a Kaggle dataset before a submission notebook can activate it",
            "default masked-prefix run is a deterministic direction-balanced subset; set MASKED_PREFIX_MAX_WELLS=0 for the required all-eligible confirmation",
            "the clean standalone spatial imputers are initialized once on all training wells; paired deltas remain isolated, but absolute OOF estimates may be optimistic because spatial priors are not refit inside each fold",
            "reduced-cost screen uses only 320 UUID-sorted wells, 32 PF seeds, 300 particles, 500 bootstrap draws, and 24 masked-prefix wells; it is never deployable",
        ],
    }

    ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
    for variant, features in (("control", control_features), ("conditioned", conditioned_features)):
        out_dir = ARTIFACT_ROOT / variant
        _fit_and_save_full_models(augmented, features, configs, best_iterations[variant], out_dir)
        manifest = dict(common_manifest)
        manifest.update({"variant": variant, "features": features,
                         "azimuth_features_active": variant == "conditioned"})
        (out_dir / "azimuth_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")

    primary_metrics.to_csv(ARTIFACT_ROOT / "paired_oof_metrics.csv", index=False)
    oof_detail.to_csv(ARTIFACT_ROOT / "paired_oof_predictions.csv", index=False)
    masked_metrics.to_csv(ARTIFACT_ROOT / "masked_prefix_metrics.csv", index=False)
    masked_detail.to_csv(ARTIFACT_ROOT / "masked_prefix_predictions.csv", index=False)
    _AZ_AXIS_AUDIT.to_csv(ARTIFACT_ROOT / "azimuth_axis_well_audit.csv", index=False)
    runtime_audit = {"builder_completed": True, "hidden_competition_rerun": False,
                     "deployable": deployable, "elapsed_seconds": time.time() - started,
                     "artifact_root": str(ARTIFACT_ROOT), "gates": gates}
    (ARTIFACT_ROOT / "builder_runtime_audit.json").write_text(json.dumps(runtime_audit, indent=2) + "\n")
    print(json.dumps(runtime_audit, indent=2))
    if not deployable:
        print("CONDITIONED ARTIFACT DID NOT PASS DEPLOYMENT GATES; do not attach it to a submission notebook.")
    return runtime_audit


In [ ]:
# Training is forbidden in a hidden code-competition rerun.
if IS_HIDDEN_COMPETITION_RERUN:
    ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
    audit = {"builder_completed": False, "hidden_competition_rerun": True,
             "reason": "artifact training disabled during hidden competition inference"}
    (ARTIFACT_ROOT / "builder_runtime_audit.json").write_text(json.dumps(audit, indent=2) + "\n")
    print("AZIMUTH ARTIFACT BUILDER SKIPPED: hidden competition rerun detected")
else:
    builder_audit = build_paired_artifacts()


## Output contract and next gate

The output directory contains `control/` and `conditioned/` artifacts, detailed paired OOF and masked-prefix reports, direction-axis audits, and a runtime audit. Each artifact has its own manifest and the exact three-LightGBM loader contract.

Do not deploy the conditioned artifact when `deployable` is false. Even after a pass, rerun with `MASKED_PREFIX_MAX_WELLS=0`, publish the artifact as a private Kaggle dataset, attach it to a separate inference-only notebook, and verify that its runtime audit reports an active finite axis and exact 200-feature order before spending a submission slot.